# Importing libaries

In [1]:
import pandas as pd
import numpy as np

# Importing data

In [2]:
df_ca = pd.read_csv('CA-species-listings-by-tax-group-report.csv')

In [3]:
df_ca.head(5)

,Scientific Name,Common Name,Where Listed,Region,ESA Listing Status,Group
0,Vagrans egistina,Mariana wandering butterfly,Wherever found,1,Endangered,Insects
1,Tryonia cheatumi,Phantom Tryonia,Wherever found,2,Endangered,Snails
2,Lepidomeda mollispinis pratensis,Big Spring spinedace,Wherever found,8,Threatened,Fishes
3,Gila bicolor ssp. mohavensis,Mohave tui chub,Wherever found,8,Endangered,Fishes
4,Tamias minimus atristriatus,Penasco least chipmunk,Wherever found,2,Endangered,Mammals


In [4]:
df_or = pd.read_csv('OR_Threatened_and_Endangered_Species.csv', header=1)

In [5]:
df_or.head(5)

,Animal Type,Common Name,Scientific Name,State Status*,Federal Status
0,FISH,Bull Trout (range-wide),Salvelinus confluentus,NaN,T
1,FISH,Columbia River Chum Salmon,Oncorhynchus keta,NaN,T
2,FISH,Green Sturgeon (Southern DPS),Acipenser medirostris,NaN,T
3,FISH,Hutton Spring Tui Chub,Siphateles bicolor ssp,T,T
4,FISH,Lahontan Cutthroat Trout,Oncorhynchus clarkii henshawi,T,T


In [6]:
df_wa = pd.read_csv('tabula-WA-state-listed-and-candidate-species-list.csv')

In [7]:
df_wa.head(5)

,Animal Type,Species,State status,Federal status
0,MAMMALS,Fin whale,Endangered,Endangered
1,MAMMALS,Sei whale,Endangered,Endangered
2,MAMMALS,Blue whale,Endangered,Endangered
3,MAMMALS,Humpback whale,Endangered,"Threatened (Mexico DPs), Endangered (Central A..."
4,MAMMALS,North Pacific Right Whale,Endangered,Endangered


# Data Cleaning

### Standardizing Columns

In [8]:
df_wa.columns

Index(['Animal Type', 'Species', 'State status', 'Federal status'], dtype='str')

In [11]:
df_or.columns

Index(['Animal Type', 'Common Name', 'Scientific Name', 'State Status*',
       'Federal Status'],
      dtype='str')

In [14]:
df_ca.columns

Index(['Scientific Name', 'Common Name', 'Where Listed', 'Region  ',
       'ESA Listing Status  ', 'Group'],
      dtype='str')

#### Animal Class: Each dataframe groups animal types slightly differently. The following will find all unique animal type or group values per data set, and create a dictionary of standardized versions. An Animal Class column will be created with the standardized names.

In [9]:
wa_animal_types = df_wa['Animal Type'].unique().tolist()

In [10]:
wa_animal_types

['MAMMALS',
 'REPTILES',
 'AMPHIBIANS',
 'INVERTEBRATES',
 'MAMMALS ',
 'BIRDS',
 'FISH ',
 'AMPHIBIAN',
 'REPTILES AND AMPHIBIANS',
 'FISH',
 'MOLLUSKS',
 'INSECTS']

In [12]:
or_animal_types = df_or['Animal Type'].unique().tolist()

In [13]:
or_animal_types

['FISH', 'AMPHIBIANS AND REPTILES', 'BIRDS', 'MAMMALS']

In [15]:
ca_group_list = df_ca['Group'].unique().tolist()

In [16]:
ca_group_list

['Insects',
 'Snails',
 'Fishes',
 'Mammals',
 'Clams',
 'Reptiles',
 'Birds',
 'Arachnids',
 'Crustaceans',
 'Amphibians']

In [17]:
animal_classes = {'Invertebrates': ['INVERTEBRATES', 'MOLLUSKS', 'Snails', 'Clams', 'Crustaceans'], 'Fish': ['Fishes', 'FISH', 'FISH '], 'Reptiles and Amphibians': ['Reptiles', 'Amphibians', 'AMPHIBIANS', 'REPTILES', 'AMPHIBIANS', 'AMPHIBIAN', 'REPTILES AND AMPHIBIANS'], 'Birds': ['Birds', 'BIRDS'], 'Mammals': ['MAMMALS', 'Mammals', 'MAMMALS '], 'Insects': ['Insects', 'Arachnids', 'INSECTS']}

In [18]:
mapping_classes = {var: std for std, vars_list in animal_classes.items() for var in vars_list}

In [19]:
df_ca['Animal Class'] = df_ca['Group'].map(mapping_classes)

In [20]:
df_ca.head(5)

,Scientific Name,Common Name,Where Listed,Region,ESA Listing Status,Group,Animal Class
0,Vagrans egistina,Mariana wandering butterfly,Wherever found,1,Endangered,Insects,Insects
1,Tryonia cheatumi,Phantom Tryonia,Wherever found,2,Endangered,Snails,Invertebrates
2,Lepidomeda mollispinis pratensis,Big Spring spinedace,Wherever found,8,Threatened,Fishes,Fish
3,Gila bicolor ssp. mohavensis,Mohave tui chub,Wherever found,8,Endangered,Fishes,Fish
4,Tamias minimus atristriatus,Penasco least chipmunk,Wherever found,2,Endangered,Mammals,Mammals


In [21]:
df_ca['Animal Class'].isna().sum()

np.int64(0)

In [22]:
df_wa['Animal Class'] = df_wa['Animal Type'].map(mapping_classes)

In [23]:
df_wa.head(5)

,Animal Type,Species,State status,Federal status,Animal Class
0,MAMMALS,Fin whale,Endangered,Endangered,Mammals
1,MAMMALS,Sei whale,Endangered,Endangered,Mammals
2,MAMMALS,Blue whale,Endangered,Endangered,Mammals
3,MAMMALS,Humpback whale,Endangered,"Threatened (Mexico DPs), Endangered (Central A...",Mammals
4,MAMMALS,North Pacific Right Whale,Endangered,Endangered,Mammals


In [24]:
df_or['Animal Class'] = df_or['Animal Type'].map(mapping_classes)

In [25]:
df_or.head(5)

,Animal Type,Common Name,Scientific Name,State Status*,Federal Status,Animal Class
0,FISH,Bull Trout (range-wide),Salvelinus confluentus,NaN,T,Fish
1,FISH,Columbia River Chum Salmon,Oncorhynchus keta,NaN,T,Fish
2,FISH,Green Sturgeon (Southern DPS),Acipenser medirostris,NaN,T,Fish
3,FISH,Hutton Spring Tui Chub,Siphateles bicolor ssp,T,T,Fish
4,FISH,Lahontan Cutthroat Trout,Oncorhynchus clarkii henshawi,T,T,Fish


#### Endangered Status: Oregon and Washington list both federal status and state status. California includes an ESA Listing Status. Will create a standard status column. 

##### Oregon

In [26]:
df_or['State Status*'].isna().sum()

np.int64(23)

In [27]:
df_or['Federal Status'].unique()

<ArrowStringArray>
['T', 'E', nan, 'T\r(Pacific Coast\rPopulation DPS)', 'T\r(Coastal DPS)', 'C']
Length: 6, dtype: str

In [28]:
status_map ={'T': 'Threatened', 'E': 'Endangered', 'NaN': 'NaN', 'T\r(Pacific Coast\rPopulation DPS)': 'Threatened', 'T\r(Coastal DPS)': 'Threatened', 'C': 'Candidate'}

In [29]:
df_or['Status'] = df_or['Federal Status'].map(status_map).fillna(df_or['State Status*'].map(status_map))

In [30]:
df_or.head(5)

,Animal Type,Common Name,Scientific Name,State Status*,Federal Status,Animal Class,Status
0,FISH,Bull Trout (range-wide),Salvelinus confluentus,NaN,T,Fish,Threatened
1,FISH,Columbia River Chum Salmon,Oncorhynchus keta,NaN,T,Fish,Threatened
2,FISH,Green Sturgeon (Southern DPS),Acipenser medirostris,NaN,T,Fish,Threatened
3,FISH,Hutton Spring Tui Chub,Siphateles bicolor ssp,T,T,Fish,Threatened
4,FISH,Lahontan Cutthroat Trout,Oncorhynchus clarkii henshawi,T,T,Fish,Threatened


In [31]:
df_or['Status'].unique()

<ArrowStringArray>
['Threatened', 'Endangered', 'Candidate']
Length: 3, dtype: str

##### California

In [32]:
ca_status_list = df_ca['ESA Listing Status  '].unique().tolist()

In [33]:
ca_status_list

['Endangered',
 'Threatened',
 'Similarity of Appearance (Threatened)',
 'Experimental Population, Non-Essential']

In [34]:
for item in ca_status_list:
    item = str(item)
    if 'Threatened' in item:
        status_map[item] = 'Threatened'
    elif 'Endangered' in item:
        status_map[item] = 'Endangered'
    elif 'Candidate' in item:
        status_map[item] = 'Candidate'
    else:
        status_map[item] = 'NaN'

In [35]:
status_map

{'T': 'Threatened',
 'E': 'Endangered',
 'NaN': 'NaN',
 'T\r(Pacific Coast\rPopulation DPS)': 'Threatened',
 'T\r(Coastal DPS)': 'Threatened',
 'C': 'Candidate',
 'Endangered': 'Endangered',
 'Threatened': 'Threatened',
 'Similarity of Appearance (Threatened)': 'Threatened',
 'Experimental Population, Non-Essential': 'NaN'}

In [36]:
df_ca['Status'] = df_ca['ESA Listing Status  '].map(status_map)

In [37]:
df_ca.head(5)

,Scientific Name,Common Name,Where Listed,Region,ESA Listing Status,Group,Animal Class,Status
0,Vagrans egistina,Mariana wandering butterfly,Wherever found,1,Endangered,Insects,Insects,Endangered
1,Tryonia cheatumi,Phantom Tryonia,Wherever found,2,Endangered,Snails,Invertebrates,Endangered
2,Lepidomeda mollispinis pratensis,Big Spring spinedace,Wherever found,8,Threatened,Fishes,Fish,Threatened
3,Gila bicolor ssp. mohavensis,Mohave tui chub,Wherever found,8,Endangered,Fishes,Fish,Endangered
4,Tamias minimus atristriatus,Penasco least chipmunk,Wherever found,2,Endangered,Mammals,Mammals,Endangered


In [38]:
df_ca['Status'].unique()

<ArrowStringArray>
['Endangered', 'Threatened', 'NaN']
Length: 3, dtype: str

##### Washington

In [39]:
df_wa['Federal status'] = df_wa['Federal status'].astype(str)

In [40]:
df_wa['Federal status'] = df_wa['Federal status'].fillna(df_wa['State status'])

In [41]:
df_wa[df_wa['Federal status'].isna()]

,Animal Type,Species,State status,Federal status,Animal Class
72,BIRDS,Nuthatch,NaN,NaN,Birds
105,INSECTS,Beetle,NaN,NaN,Insects


###### For some reason pandas is not allowing .fillna() to fix these values

In [42]:
type(df_wa.iloc[72, 3])

float

In [43]:
df_wa.iloc[72, 3]

nan

In [44]:
df_wa.iloc[72, 3] = 'Candidate'

In [45]:
df_wa.iloc[72, 3]

'Candidate'

In [46]:
df_wa.iloc[105, 3]

nan

In [47]:
df_wa.iloc[105, 3] = 'Candidate'

In [48]:
df_wa.iloc[105, 3]

'Candidate'

In [49]:
df_wa['Federal status'].isna().sum()

np.int64(0)

In [50]:
df_wa['Federal status'].unique()

<ArrowStringArray>
[                                                                                'Endangered',
                                 'Threatened (Mexico DPs), Endangered (Central American DPS)',
                                                       'Endangered (Southern Residents only)',
 'Endangered (Federally listed west of north-south line following Highways 97, 17, and 395.)',
                                                                                 'Threatened',
                                                                             'Federal status',
                                               '90-day finding that listing may be warranted',
 'Threatened. Subspecies: glacialis, pugetensis, tumuli, yelmensis, couchi, louie*, melanops',
                                                    'Endangered. Western North Pacific Stock',
                                                                                  'Sensitive',
                               

In [51]:
wa_status_list = df_wa['Federal status'].unique().tolist()
for item in wa_status_list:
    item = str(item)
    if 'Threatened' in item:
        status_map[item] = 'Threatened'
    elif 'Endangered' in item:
        status_map[item] = 'Endangered'
    elif 'Candidate' in item:
        status_map[item] = 'Candidate'
    elif 'Sensitive' in item:
        status_map[item] = 'Sensitive'
    else:
        status_map[item] = pd.NA

In [52]:
df_wa['Status'] = df_wa['Federal status'].map(status_map).fillna(df_wa['State status'].map(status_map))

In [53]:
df_wa['Status'].unique()

<ArrowStringArray>
['Endangered', 'Threatened', nan, 'Sensitive', 'Candidate']
Length: 5, dtype: str

In [54]:
df_wa['Status'].isna().sum()

np.int64(3)

In [55]:
df_wa[df_wa['Status'].isna()]

,Animal Type,Species,State status,Federal status,Animal Class,Status
22,MAMMALS,Species,State status,Federal status,Mammals,NaN
38,INVERTEBRATES,Species,State status,Federal status,Invertebrates,NaN
92,FISH,Species,State status,Federal status,Fish,NaN


In [56]:
df_wa = df_wa.dropna()

In [57]:
df_wa['Status'].isna().sum()

np.int64(0)

#### WA 'Species' is most similar to Common Name.

In [59]:
df_wa.head(5)

,Animal Type,Species,State status,Federal status,Animal Class,Status
0,MAMMALS,Fin whale,Endangered,Endangered,Mammals,Endangered
1,MAMMALS,Sei whale,Endangered,Endangered,Mammals,Endangered
2,MAMMALS,Blue whale,Endangered,Endangered,Mammals,Endangered
3,MAMMALS,Humpback whale,Endangered,"Threatened (Mexico DPs), Endangered (Central A...",Mammals,Threatened
4,MAMMALS,North Pacific Right Whale,Endangered,Endangered,Mammals,Endangered


In [60]:
df_wa = df_wa.rename(columns={'Species': 'Common Name'})

In [61]:
df_wa.head()

,Animal Type,Common Name,State status,Federal status,Animal Class,Status
0,MAMMALS,Fin whale,Endangered,Endangered,Mammals,Endangered
1,MAMMALS,Sei whale,Endangered,Endangered,Mammals,Endangered
2,MAMMALS,Blue whale,Endangered,Endangered,Mammals,Endangered
3,MAMMALS,Humpback whale,Endangered,"Threatened (Mexico DPs), Endangered (Central A...",Mammals,Threatened
4,MAMMALS,North Pacific Right Whale,Endangered,Endangered,Mammals,Endangered


#### Adding State Column

In [70]:
df_wa['State'] = 'Washington'

In [72]:
df_or['State'] = 'Oregon'

In [73]:
df_ca['State'] = 'California'

In [63]:
df_wa = df_wa.astype(str)

In [65]:
df_or = df_or.astype(str)

In [66]:
df_ca = df_ca.astype(str)

In [74]:
df_wa.info()

<class 'pandas.DataFrame'>
Index: 117 entries, 0 to 121
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   Animal Type     117 non-null    str  
 1   Common Name     117 non-null    str  
 2   State status    117 non-null    str  
 3   Federal status  117 non-null    str  
 4   Animal Class    117 non-null    str  
 5   Status          117 non-null    str  
 6   State           117 non-null    str  
dtypes: str(7)
memory usage: 16.3 KB


In [ ]:
df_wa = df_wa[

In [75]:
df_ca.info()

<class 'pandas.DataFrame'>
RangeIndex: 749 entries, 0 to 748
Data columns (total 9 columns):
 #   Column                Non-Null Count  Dtype
---  ------                --------------  -----
 0   Scientific Name       749 non-null    str  
 1   Common Name           749 non-null    str  
 2   Where Listed          738 non-null    str  
 3   Region                749 non-null    str  
 4   ESA Listing Status    749 non-null    str  
 5   Group                 749 non-null    str  
 6   Animal Class          749 non-null    str  
 7   Status                749 non-null    str  
 8   State                 749 non-null    str  
dtypes: str(9)
memory usage: 136.8 KB


In [76]:
df_or.info()

<class 'pandas.DataFrame'>
RangeIndex: 53 entries, 0 to 52
Data columns (total 8 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   Animal Type      53 non-null     str  
 1   Common Name      53 non-null     str  
 2   Scientific Name  53 non-null     str  
 3   State Status*    30 non-null     str  
 4   Federal Status   49 non-null     str  
 5   Animal Class     48 non-null     str  
 6   Status           53 non-null     str  
 7   State            53 non-null     str  
dtypes: str(8)
memory usage: 7.3 KB


### Merge Datasets

In [77]:
columns = {'Animal Class', 'Common Name', 'Scientific Name', 'Status', 'State'}

In [79]:
df1 = df_ca[['Animal Class', 'Common Name', 'Scientific Name', 'Status', 'State']]

In [82]:
df2 = df_wa[['Animal Class', 'Common Name', 'Status', 'State']]

In [83]:
df3 = df_or[['Animal Class', 'Common Name', 'Scientific Name', 'Status', 'State']]

In [88]:
df_pnw = pd.concat([df3, df2], ignore_index=True)

In [90]:
df_pnw

,Animal Class,Common Name,Scientific Name,Status,State
0,Fish,Bull Trout (range-wide),Salvelinus confluentus,Threatened,Oregon
1,Fish,Columbia River Chum Salmon,Oncorhynchus keta,Threatened,Oregon
2,Fish,Green Sturgeon (Southern DPS),Acipenser medirostris,Threatened,Oregon
3,Fish,Hutton Spring Tui Chub,Siphateles bicolor ssp,Threatened,Oregon
4,Fish,Lahontan Cutthroat Trout,Oncorhynchus clarkii henshawi,Threatened,Oregon
...,...,...,...,...,...
165,Insects,Valley Silverspot,NaN,Candidate,Washington
166,Insects,Silver-bordered Fritillary,NaN,Candidate,Washington
167,Insects,Monarch,NaN,Candidate,Washington
168,Insects,Great Arctic,NaN,Candidate,Washington


In [91]:
df_species = pd.concat([df_pnw, df1], ignore_index=True)

In [92]:
df_species

,Animal Class,Common Name,Scientific Name,Status,State
0,Fish,Bull Trout (range-wide),Salvelinus confluentus,Threatened,Oregon
1,Fish,Columbia River Chum Salmon,Oncorhynchus keta,Threatened,Oregon
2,Fish,Green Sturgeon (Southern DPS),Acipenser medirostris,Threatened,Oregon
3,Fish,Hutton Spring Tui Chub,Siphateles bicolor ssp,Threatened,Oregon
4,Fish,Lahontan Cutthroat Trout,Oncorhynchus clarkii henshawi,Threatened,Oregon
...,...,...,...,...,...
914,Insects,Quino checkerspot butterfly,Euphydryas editha quino (=E. e. wrighti),Endangered,California
915,Invertebrates,Appalachian elktoe,Alasmidonta raveneliana,Endangered,California
916,Mammals,Santa Catalina Island Fox,Urocyon littoralis catalinae,Threatened,California
917,Reptiles and Amphibians,Barton Springs salamander,Eurycea sosorum,Endangered,California


In [94]:
df_species = df_species.groupby('Common Name', as_index=False).agg({'State': ' ,'.join, 'Scientific Name': 'first', 'Status': 'first'})

In [95]:
df_species

,Common Name,State,Scientific Name,Status
0,Akekee,California,Loxops caeruleirostris,Endangered
1,Akikiki,California,Oreomystis bairdi,Endangered
2,Alabama Map turtle,California,Graptemys pulchra,Threatened
3,Alabama beach mouse,California,Peromyscus polionotus ammobates,Endangered
4,Alabama cave shrimp,California,Palaemonias alabamae,Endangered
...,...,...,...,...
786,salt marsh harvest mouse,California,Reithrodontomys raviventris,Endangered
787,shortnose sucker,California,Chasmistes brevirostris,Endangered
788,tidewater goby,California,Eucyclogobius newberryi,Endangered
789,unarmored threespine stickleback,California,Gasterosteus aculeatus williamsoni,Endangered


In [97]:
df_species.to_csv('clean_west_coast_endangered_species.csv', encoding='utf-8-sig', index=False)